In [110]:
import numpy as np
import pandas as pd

In [111]:
from random import randint
from scipy.spatial.transform import Rotation as R

def rotate_thf(thf):
    """
    This function randomizes the orientation of THF in space. It does this with a transform matrix
    provided by the scipy Rotation package. By using this package, THF can be rotated around it's
    center in 3D space, and not just around an axis. This funtion also provides random rotations,
    with a random number of degrees in each direction.

    For more information on the theory, visit:
    https://stackoverflow.com/questions/14607640/rotating-a-vector-in-3d-space

    Input: THF dataframe that includes columns for 'X', 'Y', and 'Z'

    Output: Identical THF dataframe with updated coordinates given a random rotation in 3D space

    Author: Gabe Miles
    """

    x,y,z = randint(0,360), randint(0,360), randint(0,360)
    r = R.from_euler('zyx', [x,y,z], degrees=True)
    tmp = np.empty((0,3), int)
    for i in range(len(thf.index)):
        # np.vstack returns the column vector of 'zyx', and the '@' does matrix multiplication
        new_xyz = r.as_matrix() @ np.vstack(thf[['X','Y','Z']].iloc[i].values)
        tmp = np.append(tmp, new_xyz.T, axis=0)

    rot_thf = pd.DataFrame({'ELEMENT': thf['ELEMENT'],
                        'X': tmp[:,[0]].T[0],
                        'Y': tmp[:,[1]].T[0],
                        'Z': tmp[:,[2]].T[0],
                        'RESIDUE_NUM': thf['RESIDUE_NUM'],
                        'MOL': thf['MOL']})
    
    return rot_thf

In [112]:
# pdb=pd.read_csv('output/low_hydrate.pdb', header=0, names=['TYPE','ID','ELEMENT','MOL','CHAIN_ID','RESIDUE_NUM','X','Y','Z','NUM1','NUM2'], delim_whitespace=True)
rando_hydrate = pd.read_csv('output/low_hydrate.csv')
cages=pd.read_csv('input/s2-cages.csv')
thf=pd.read_csv('input/thf.csv')
rando_hydrate


,ELEMENT,MOL,RESIDUE_NUM,X,Y,Z
0,OW,HOH,1,5.924870,5.924870,5.924870
1,HW1,HOH,1,5.772569,5.145594,6.459433
2,HW2,HOH,1,5.986392,5.593145,5.029095
3,MW,HOH,1,5.913249,5.782648,5.878631
4,OW,HOH,2,5.321790,5.321790,8.566200
...,...,...,...,...,...,...
14679,MW,HOH,3670,30.243477,47.449164,-12.996931
14680,OW,HOH,3671,21.627480,47.593350,-4.306710
14681,HW1,HOH,3671,21.317733,46.694990,-4.421725
14682,HW2,HOH,3671,22.237375,47.544520,-3.570586


In [113]:
# index = len(rando_hydrate.index)
# rando_hydrate.loc[3997:index, ['X','Y','Z']] = pdb.loc[3997:index, ['RESIDUE_NUM','X','Y']].values

# # Hydrate has incorrect residue numbers, and they are floats
# rando_hydrate = rando_hydrate.drop(['RESIDUE_NUM'], axis=1)
# rando_hydrate['RESIDUE_NUM'] = 0
# residue_num = 1
# for i in range(0,len(rando_hydrate.index)):
#     rando_hydrate.loc[i,'RESIDUE_NUM'] = residue_num
#     if rando_hydrate.loc[i,'ELEMENT'] == 'MW':
#         residue_num += 1
# # = pdb[['RESIDUE_NUM','X','Y']].iloc[3997:len(rando_hydrate.index)].values
# rando_hydrate.max()

In [114]:
# f = open("output/rmp.pdb", "w")

# f.write("CRYST1   52.000   52.000   52.000  90.00  90.00  90.00 P 1           1\n")
# for i in range(len(rando_hydrate.index)):
#     #Collect PDB variables
#     id = str(i)
#     chainid = 'A'
#     atom = rando_hydrate['ELEMENT'].iloc[i]
#     resname = rando_hydrate['MOL'].iloc[i]
#     resnum = rando_hydrate['RESIDUE_NUM'].iloc[i]
#     x = rando_hydrate['X'].iloc[i]
#     y = rando_hydrate['Y'].iloc[i]
#     z = rando_hydrate['Z'].iloc[i]

#     # Create PDB entry
#     line = "{:6s}{:5d} {:^4s} {:3s} {:1s}{:5d}    {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}\n".format(
#            "ATOM", i+1, atom, resname, chainid, resnum, x, y, z, 0., 0.)
#     f.write(line)

# f.close()

In [115]:
# Insert THF
large_cages = cages[cages['TYPE'].str.contains('Large')]
thf_center = thf[['X', 'Y', 'Z']].mean()
# Set center to (0,0,0)
thf[['X','Y','Z']] = thf[['X','Y','Z']] - thf_center
# Update to new center
thf['RESIDUE_NUM'] = -1
thf['MOL'] = 'THF'

In [116]:
#Make the box 3 x 3
cell_size = 17.30
num_cells = 3
coord_offset = [[-cell_size, 0, cell_size],[0,0,cell_size],[cell_size, 0, cell_size],
                [-cell_size, 0, 0],[cell_size, 0, 0],
                [-cell_size, 0, -cell_size],[0, 0, -cell_size],[cell_size, 0, -cell_size]]

In [117]:
cages_3_plane = large_cages.copy()
for offset in coord_offset:
    # For hydrate
    large_cages_unit = large_cages.copy()
    large_cages_unit[['X','Y','Z']] = large_cages_unit[['X','Y','Z']] + offset
    cages_3_plane = pd.concat([cages_3_plane, large_cages_unit], axis=0, ignore_index=True)

In [118]:
# Stack 3x3x3 crystal
cages_3_3 = cages_3_plane.copy()
for i in range(1, num_cells):
    cages_unit = cages_3_plane.copy()
    cages_unit['Y'] = cages_unit['Y'] + (cell_size * i)
    cages_3_3 = pd.concat([cages_3_3, cages_unit], axis=0, ignore_index=True)

In [119]:
rando_hydrate.info

<bound method DataFrame.info of       ELEMENT  MOL  RESIDUE_NUM          X          Y          Z
0          OW  HOH            1   5.924870   5.924870   5.924870
1         HW1  HOH            1   5.772569   5.145594   6.459433
2         HW2  HOH            1   5.986392   5.593145   5.029095
3          MW  HOH            1   5.913249   5.782648   5.878631
4          OW  HOH            2   5.321790   5.321790   8.566200
...       ...  ...          ...        ...        ...        ...
14679      MW  HOH         3670  30.243477  47.449164 -12.996931
14680      OW  HOH         3671  21.627480  47.593350  -4.306710
14681     HW1  HOH         3671  21.317733  46.694990  -4.421725
14682     HW2  HOH         3671  22.237375  47.544520  -3.570586
14683      MW  HOH         3671  21.665903  47.472098  -4.227201

[14684 rows x 6 columns]>

In [120]:
# This loop recenters THF and places it in all 8 large cages of the sII hydrate
thf_hydrate_randomized_3_3 = rando_hydrate.copy()
for i in range(len(cages_3_3.index)):
    rot_thf = rotate_thf(thf)
    dxyz = thf_center - cages_3_3[['X','Y','Z']].iloc[i]
    thf_translated = rot_thf.copy()
    thf_translated[['X','Y','Z']] = thf_translated[['X','Y','Z']] - dxyz
    thf_translated['RESIDUE_NUM'] = thf_hydrate_randomized_3_3['RESIDUE_NUM'].iloc[len(thf_hydrate_randomized_3_3.index) - 1] + 1
    thf_hydrate_randomized_3_3 = pd.concat([thf_hydrate_randomized_3_3, thf_translated], ignore_index=True)
    residue_num += 1

thf_hydrate_randomized_3_3

,ELEMENT,MOL,RESIDUE_NUM,X,Y,Z
0,OW,HOH,1,5.924870,5.924870,5.924870
1,HW1,HOH,1,5.772569,5.145594,6.459433
2,HW2,HOH,1,5.986392,5.593145,5.029095
3,MW,HOH,1,5.913249,5.782648,5.878631
4,OW,HOH,2,5.321790,5.321790,8.566200
...,...,...,...,...,...,...
17487,H08,THF,3887,17.884539,41.902805,-18.417700
17488,H09,THF,3887,18.219647,43.079920,-15.638228
17489,H0A,THF,3887,17.225571,41.682954,-16.094533
17490,H0B,THF,3887,16.123370,44.177747,-15.912401


In [121]:
f = open("output/tip4p_thf_hydrate_randomized_0_dipole.pdb", "w")

f.write("CRYST1   52.000   52.000   52.000  90.00  90.00  90.00 P 1           1\n")
for i in range(len(thf_hydrate_randomized_3_3.index)):
    #Collect PDB variables
    id = str(i)
    chainid = 'A'
    atom = thf_hydrate_randomized_3_3['ELEMENT'].iloc[i]
    resname = thf_hydrate_randomized_3_3['MOL'].iloc[i]
    resnum = thf_hydrate_randomized_3_3['RESIDUE_NUM'].iloc[i]
    x = thf_hydrate_randomized_3_3['X'].iloc[i]
    y = thf_hydrate_randomized_3_3['Y'].iloc[i]
    z = thf_hydrate_randomized_3_3['Z'].iloc[i]

    # Create PDB entry
    line = "{:6s}{:5d} {:^4s} {:3s} {:1s}{:5d}    {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}\n".format(
           "ATOM", i+1, atom, resname, chainid, resnum, x, y, z, 0., 0.)
    f.write(line)

f.close()